# Part 5: Evaluation

A training run does not produce an accuracy number. It produces a checkpoint. The number
comes later, from a separate procedure with its own gates, and no AUC appears in any
document of this project until that procedure has run to the end.

This part covers the path from checkpoint to trustworthy number: the metric, which of
the two AUCs is the number of record, the scoring job, the recomputation gate every
number must pass, and the uncertainty analysis that decides whether a difference between
two numbers is real.

## The metric

The headline metric is ROC AUC — the probability that a randomly chosen signal jet
scores higher than a randomly chosen background jet. Its defining property here is that
it is **threshold-free**. The trigger will eventually run at some working point chosen
against a rate budget that does not exist yet; a single-threshold metric like accuracy
would prejudge that choice, while AUC measures the ranking quality any future working
point will cut on.

With five jet classes (gluon, light quark, W, Z, top) we report the **macro one-vs-rest
AUC**: a binary AUC for each class against the other four, then the unweighted mean of
the five. Unweighted is deliberate — it is insensitive to class balance and privileges no
class, so a model cannot buy headline accuracy by sacrificing the rarest signature.

## Two AUCs, never conflated

**Validation AUC** is the training monitor. It is computed during training on a
validation split carved out of the *training* data (a 0.2 fraction) and it does two
jobs: select the best epoch, drive early stopping. Because model selection optimizes it
directly, it is biased upward as an estimate of generalization. It is a control signal,
not a result.

**ROC-test AUC** is the number of record: computed on the dataset's own held-out split,
n = 260,000 jets, never touched by any training or selection step. Every table states
which AUC it holds; the two never share a column.

In [ ]:
src = open("code/roc_final.py").read().splitlines()
i = next(k for k, l in enumerate(src) if l.startswith("def load_data_checked"))
print("\n".join(src[i:i + 10]))

The count is enforced, not assumed. The evaluation driver takes `--expect-n 260000` as a
hard gate and aborts on any other value, because a different count means a different
split, and a number from a different split would be silently incomparable with every
table in the project.

## The ROC job

Scoring runs as its own cluster job, whose payload is one command: `roc_final.py
eval-all`. The driver downloads the held-out split, loads all 260,000 jets into memory
**once**, and then loops over the round's (variant, seed) matrix.

In [ ]:
yaml = open("code/jobs/kai-bn14-roc-n8.yaml").read().splitlines()
i = next(k for k, l in enumerate(yaml) if "roc_final.py" in l and "eval-all" in l)
print("\n".join(l.strip() for l in yaml[i:i + 8]))

For each model the driver fetches the checkpoint of record from its W&B artifact (part 4)
and applies *that checkpoint's own* `input_std.json` before scoring.

In [ ]:
i = next(k for k, l in enumerate(src) if "Round-8 input_std contract" in l)
print("\n".join(l[16:] for l in src[i:i + 9]))

This step is easy to get wrong and fatal when wrong: the standardization statistics are
part of the model, so a checkpoint scored under another model's statistics computes a
plausible-looking, incorrect number.

Each model yields one `.npz` file with exactly three keys — `y` (one-hot labels, N×5),
`score` (softmax outputs, N×5), and `meta` (a JSON provenance string) — landing in the
round's store with a summary table `roc_auc.md` and an overlay plot beside them. Those
`.npz` files are the primary artifact of the entire accuracy story: they hold per-jet
scores, not summaries, so any AUC, any ROC curve, and any bootstrap can be recomputed
from them indefinitely, by anyone, without a GPU.

## The verification gate

The non-negotiable rule: before any AUC is written into any document — the living
research log, a frozen report, a slide, these notebooks — it is recomputed independently
from the `.npz`.

In [ ]:
# runs on: laptop, in the research tree where the ROC arrays live
import numpy as np
from sklearn.metrics import roc_auc_score

d = np.load("bnjettag/roc-results/r14/n8/W1A8-s3.npz")
assert d["y"].shape == (260_000, 5)      # wrong n = wrong split = not comparable

per_class = [roc_auc_score(d["y"][:, c], d["score"][:, c]) for c in range(5)]
print("per-class OvR AUC :", [round(a, 4) for a in per_class])
print("macro-OvR AUC     :", round(float(np.mean(per_class)), 4))

That array — round 14, $N = 8$, W1A8, seed 3 — recomputes to a ROC-test macro-OvR AUC of
**0.8724** (n = 260,000), matching what
`bnjettag/roc-results/r14/n8/roc_auc.md` claims for it.

The recomputed value is then compared against *every* place the number is claimed. Any
mismatch is a finding to be chased to its root, never rounded away or quietly fixed.

(The `.npz` arrays live in the research tree's result store, not in this repository, so
that cell is the one code cell here a reader cannot run as-is.)

The gate earns its keep in both directions. Round 14's sixty models were verified 60/60
exact on 2026-08-03 before the round's table entered the record. A pass over the
capacity-ladder tables on 2026-07-26 went the other way: four corrections and one
withdrawn claim. Both outcomes are the gate working.

For a sense of what verified numbers look like — the FINAL campaign (era-2, `large`
scale, 16 offline-style features), ROC-test macro-OvR AUC, n = 260,000, mean ± sample
std over 3 seeds (`bnjettag/roc-results/final/roc_auc.md`):

| variant | W/A bits | ROC-test macro-OvR AUC |
|---|---|---|
| FP32 | 32/32 | 0.8561 ± 0.0021 |
| W8A8 | 8/8 | 0.8561 ± 0.0007 |
| W1A8 | 1/8 | 0.8503 ± 0.0054 |
| W1A6 | 1/6 | 0.8363 ± 0.0023 |
| W1A4 | 1/4 | 0.7612 ± 0.0015 |

Those numbers belong to that campaign only. Round 14's numbers (`small` scale, three
L1-realistic features) are a different campaign on a different input set and never share
a table with them.

![FINAL-campaign accuracy ladder: ROC-test macro-OvR AUC (n = 260,000) per precision variant, individual seeds behind the seed mean ± std. Plotted from the verified per-seed table in `bnjettag/roc-results/final/roc_auc.md`.](figures/accuracy_ladder.png)

## Is the difference real?

A gap between two variants is never reported without an interval, because two noise
sources sit under every comparison.

**Training variance.** The same config at three seeds gives three different models; the
sample standard deviation over seeds measures how much of a gap is initialization luck.

**Finite-test-set variance.** Even a fixed pair of models would show a different gap on a
different draw of 260,000 jets.

The second term is estimated with a **paired** bootstrap: resample the jets with
replacement, recompute *both* models' macro AUC on each replicate, and take the standard
deviation of the *difference*.

The pairing is the point. Both models score the same jets, so jet-level difficulty is
common to both and cancels in the difference, which gives a far tighter and more honest
interval than two independent error bars.

The verdict rule combines the two sources. A gap is **resolved** when it exceeds
$2\sigma_{total}$, with

$$\sigma_{total}^2 = \frac{\sigma_{seed,A}^2}{3} + \frac{\sigma_{seed,B}^2}{3} + \sigma_{boot}^2 .$$

Round 14 at $N = 8$ shows the machinery earning conclusions
(`bnjettag/results/r14/uncertainty_r14.md`):

| comparison | gap in ROC-test macro-OvR AUC | $\sigma_{total}$ | verdict |
|---|---|---|---|
| fp32 − w1a8 | +0.0152 | 0.0010 | resolved — the binarization cost is real there |
| fp32 − w8a8 | +0.0002 | 0.0006 | unresolved — indistinguishable from zero |

The statement "8-bit weights are free at $N = 8$" is licensed *only* because that second
interval says so.

One structural fact shapes the whole comparison budget: with 260,000 test jets,
$\sigma_{boot}$ is typically a few $10^{-4}$ while seed spread runs several times larger.
Training variance, not the test set, is what limits resolution — which is why the
three-seed policy of part 4 is not negotiable.

![Round-14 ROC-test macro-OvR AUC versus constituent count N per quantization arm, seed mean ± sample std over 3 seeds. Generated by `bnjettag/code/plots/make_r14_plots.py` from the verified arrays in `bnjettag/roc-results/r14/`.](figures/fig_r14_auc_vs_n.png)

## The plot convention

Every ROC curve in this project is drawn the same way: tagging efficiency (true-positive
rate) on a **linear** x-axis, mistag rate (false-positive rate) on a **logarithmic**
y-axis.

The reason is physical rather than aesthetic. Trigger operating points live at mistag
rates spanning $10^{-1}$ down to $10^{-4}$, and on a linear axis everything below a few
percent mistag — the entire region the trigger cares about — collapses into an invisible
sliver along the floor. The log axis spends its resolution where the physics is.

![FINAL-campaign ROC curves per class (g, q, W, Z, t vs rest) in the HEP convention, efficiency linear and mistag rate logarithmic, all five precision variants on the held-out split. Generated by `bnjettag/code/hgq2/roc_final.py` from `bnjettag/roc-results/final/`.](figures/roc_overlay.png)

The chain does not end when the number is verified. The same checkpoints continue into
the hardware flow, where a different kind of gate takes over: part 6.